# Phase 1 - Playground
## Part 1: Tokenization (lightweight demo)

This notebook makes **tokenization** visible: how the text you type becomes the numbers a model actually reads.

We use `tiktoken` (OpenAI's tokenizer) because it is tiny and already installed - no model download needed. **Embeddings** (Part 2) are explained as a concept here and shown with a real model in a later step.

**How to run:** pick the `Python 3 (.venv)` kernel (top-right in VS Code), then run each cell top to bottom.

### 1. Load a tokenizer
`cl100k_base` is the tokenizer used by GPT-4 and `gpt-4o-mini`. It splits text into **sub-word** tokens - not whole words.

In [1]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")  # GPT-4 / gpt-4o-mini tokenizer

def show_tokens(text):
    """Print how `text` is broken into tokens."""
    ids = enc.encode(text)
    pieces = [enc.decode([i]) for i in ids]   # decode each id back to its piece
    print(f"Text        : {text!r}")
    print(f"Token count : {len(ids)}")
    print(f"Token IDs   : {ids}")
    print(f"Token pieces: {pieces}")

show_tokens("Hello, LLM!")

Text        : 'Hello, LLM!'
Token count : 5
Token IDs   : [9906, 11, 445, 11237, 0]
Token pieces: ['Hello', ',', ' L', 'LM', '!']


### 2. Notice the patterns
Watch how a leading space attaches to a word, how common words are a single token, and how rare or long words split into several pieces.

In [2]:
examples = [
    "cat",
    "cats",
    "kitten",
    "antidisestablishmentarianism",
    "ChatGPT is amazing!",
    "def add(a, b): return a + b",
    "Hola, mundo!",
]
for text in examples:
    show_tokens(text)
    print("-" * 70)

Text        : 'cat'
Token count : 1
Token IDs   : [4719]
Token pieces: ['cat']
----------------------------------------------------------------------
Text        : 'cats'
Token count : 1
Token IDs   : [38552]
Token pieces: ['cats']
----------------------------------------------------------------------
Text        : 'kitten'
Token count : 2
Token IDs   : [74, 23257]
Token pieces: ['k', 'itten']
----------------------------------------------------------------------
Text        : 'antidisestablishmentarianism'
Token count : 6
Token IDs   : [519, 85342, 34500, 479, 8997, 2191]
Token pieces: ['ant', 'idis', 'establish', 'ment', 'arian', 'ism']
----------------------------------------------------------------------
Text        : 'ChatGPT is amazing!'
Token count : 6
Token IDs   : [16047, 38, 2898, 374, 8056, 0]
Token pieces: ['Chat', 'G', 'PT', ' is', ' amazing', '!']
----------------------------------------------------------------------
Text        : 'def add(a, b): return a + b'
Token count

### 3. Tokens are not words (why cost & limits care)
API price and the **context window** are measured in **tokens**, not words. Long words, code, emoji, and non-English text often use *more* tokens than you would guess.

In [3]:
samples = {
    "short English": "The quick brown fox jumps.",
    "one long word": "antidisestablishmentarianism",
    "python code"  : "for i in range(10): print(i)",
    "with emoji"   : "I love AI robots",
    "spanish"      : "El zorro rapido salta.",
}
print(f"{'sample':15} | {'words':>5} | {'tokens':>6}")
print("-" * 34)
for name, text in samples.items():
    words = len(text.split())
    tokens = len(enc.encode(text))
    print(f"{name:15} | {words:5} | {tokens:6}")

sample          | words | tokens
----------------------------------
short English   |     5 |      6
one long word   |     1 |      6
python code     |     5 |     10
with emoji      |     4 |      4
spanish         |     4 |      8


### 4. Part 2: Embeddings (concept)

After tokenization, each token id is turned into an **embedding** - a vector of numbers that captures meaning. We keep the heavy model download out of this notebook, so here is the idea:

- Every token becomes a list of numbers (e.g. 768 of them).
- Tokens with similar meaning have vectors pointing in similar directions.
- That is why `king - man + woman` is approximately `queen`: it is literally vector math.

We will load a real Hugging Face model and print actual embedding vectors + a similarity score in a later step (once we install `transformers` + `torch`).

**Takeaway:** text -> tokens (numbers) -> embeddings (meaning as vectors) -> the model reasons over those vectors.